# Two-Tower Recommender — Final Training

Production training notebook for model-only deployment via Feature Views → Vector Search.

Key design decisions from the evaluation audit:
* **v2 architecture**: shared embeddings, sequence encoding, 12 user numerics
* **logQ frequency correction**: fixes biased in-batch softmax (the root cause of the MLP gap)
* **Duplicate masking**: prevents same-item rows from acting as negatives
* **Held-out evaluation**: Nov 1–3 eval, Nov 4–6 test

A co-occurrence RRF blend example is available in `04_cooccurrence_blend`.

Requires GPU compute. Run `02_feature_definitions` first to create training/eval/test tables.

In [0]:
%pip install databricks-feature-engineering>=0.16.0 databricks-vectorsearch torch scipy --quiet
dbutils.library.restartPython()

In [0]:
import os, sys, yaml, json, math, time as _time, random, glob, pickle, shutil, importlib
import numpy as np
import pandas as pd
import mlflow
import torch
import torch.nn as nn
import torch.nn.functional as TF
from torch.utils.data import IterableDataset, DataLoader
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, DoubleType, IntegerType, ArrayType, FloatType,
)
import pyarrow.parquet as pq

config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
PRODUCT_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_train_table']}"
SESSION_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['session_train_table']}"
EVALUATION_TABLE = f"{CATALOG}.{SCHEMA}.{config['eval_table']}"
TEST_TABLE = f"{CATALOG}.{SCHEMA}.{config.get('test_table', 'two_tower_fv_test')}"
EMBED_DIM = config["embed_dim"]
BATCH_SIZE = config["batch_size"]
NUM_EPOCHS = config["num_epochs"]

VOLUME_BASE = config.get("volume_base", f"/Volumes/{config.get('volume_catalog', 'shared')}/{config.get('volume_schema', 'mlc_schema')}/{config.get('volume', 'recommender_data')}")
TRAIN_PARQUET_PATH = f"{VOLUME_BASE}/session_train"
EVAL_PARQUET_PATH = f"{VOLUME_BASE}/session_eval"
TEST_PARQUET_PATH = f"{VOLUME_BASE}/session_test"
NUM_PARQUET_FILES = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mlflow.set_experiment(config["mlflow_experiment"])

print(f"Catalog:   {CATALOG}")
print(f"Schema:    {SCHEMA}")
print(f"Embed dim: {EMBED_DIM}, Batch: {BATCH_SIZE}, Epochs: {NUM_EPOCHS}")
print(f"Device:    {device}")

In [0]:
# ---- Read persisted training/eval/test tables (from 02_feature_definitions) ----
_prod = spark.table(PRODUCT_TRAIN_TABLE)
_sess = spark.table(SESSION_TRAIN_TABLE)
_common = sorted(set(_prod.columns) & set(_sess.columns))
_sess_only = [c for c in _sess.columns if c not in set(_prod.columns)]
train_df = _prod.join(_sess.select(*_common, *_sess_only), on=_common, how="inner")
eval_df = spark.table(EVALUATION_TABLE)
test_df = spark.table(TEST_TABLE)
print(f"Train: {train_df.count():,} rows")
print(f"Eval:  {eval_df.count():,} rows")
print(f"Test:  {test_df.count():,} rows")

# ---- Build vocabulary maps from TRAIN only (0 = <PAD>/<UNK>) ----
def build_vocab(sdf, col):
    vals = [row[col] for row in sdf.select(col).distinct().collect() if row[col] is not None]
    return {v: i + 1 for i, v in enumerate(sorted(vals))}

item_vocab = build_vocab(train_df, "product_id")
category_vocab = build_vocab(train_df, "item_category_l1")
category_l2_vocab = build_vocab(train_df, "item_category_l2")
category_l3_vocab = build_vocab(train_df, "item_category_l3")
brand_vocab = build_vocab(train_df, "item_brand")

vocabs = {
    "item_vocab": item_vocab,
    "category_vocab": category_vocab,
    "category_l2_vocab": category_l2_vocab,
    "category_l3_vocab": category_l3_vocab,
    "brand_vocab": brand_vocab,
}
print(f"Vocabs — items: {len(item_vocab)}, cat_l1: {len(category_vocab)}, "
      f"cat_l2: {len(category_l2_vocab)}, cat_l3: {len(category_l3_vocab)}, "
      f"brands: {len(brand_vocab)}")

In [0]:
# ---------------------------------------------------------------------------
# Feature transforms: item features + user features (scalars + sequences).
# Applied via mapInPandas on Spark executors for distributed processing.
# ---------------------------------------------------------------------------
MAX_RECENT_PRODUCTS = 10
MAX_RECENT_BRANDS = 5

def pad_sequence(seq, max_len, pad_val=0):
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return [pad_val] * max_len
    seq = list(seq)[:max_len]
    return seq + [pad_val] * (max_len - len(seq))

def make_mask(seq, max_len):
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return [0.0] * max_len
    n = min(len(list(seq)), max_len)
    return [1.0] * n + [0.0] * (max_len - n)

def transform_item_features(pdf, vocabs):
    df = pdf.copy()
    df["item_log_price"] = df["item_log_price"].fillna(0.0)
    df["item_popularity"] = np.log1p(df["item_view_count"].fillna(0))
    df["item_conversion_rate"] = (
        (df["item_purchase_count"].fillna(0) + 1) / (df["item_view_count"].fillna(0) + 10)
    )
    df["item_cart_to_purchase_rate"] = (
        (df["item_purchase_count"].fillna(0) + 1) / (df["item_cart_count"].fillna(0) + 10)
    )
    cat1_range = (df["category_l1_max_price"] - df["category_l1_min_price"]).clip(lower=1e-6)
    df["price_pct_rank"] = ((df["item_price"] - df["category_l1_min_price"]) / cat1_range).fillna(0.5)
    cat2_range = (df["category_l2_max_price"] - df["category_l2_min_price"]).clip(lower=1e-6)
    df["price_pct_rank_l2"] = ((df["item_price"] - df["category_l2_min_price"]) / cat2_range).fillna(0.5)
    if "event_time" in df.columns and "item_recency" in df.columns:
        event_dates = pd.to_datetime(df["event_time"]).dt.normalize()
        recency_dates = pd.to_datetime(df["item_recency"]).dt.normalize()
        df["item_recency_val"] = (1.0 / (1.0 + (event_dates - recency_dates).dt.days)).fillna(0.0)
    else:
        df["item_recency_val"] = 0.0
    df["item_idx"] = df["product_id"].map(vocabs["item_vocab"]).fillna(0).astype(int)
    df["category_idx"] = df["item_category_l1"].map(vocabs["category_vocab"]).fillna(0).astype(int)
    df["category_l2_idx"] = df["item_category_l2"].map(vocabs["category_l2_vocab"]).fillna(0).astype(int)
    df["category_l3_idx"] = df["item_category_l3"].map(vocabs["category_l3_vocab"]).fillna(0).astype(int)
    df["brand_idx"] = df["item_brand"].map(vocabs["brand_vocab"]).fillna(0).astype(int)
    return df

def transform_user_features(pdf, vocabs):
    df = pdf.copy()
    for col in ["session_event_count", "session_avg_price", "session_price_stddev",
                "session_total_value", "session_cart_value",
                "session_view_count", "session_cart_count", "session_purchase_count"]:
        df[col] = pd.to_numeric(df.get(col, 0), errors="coerce").fillna(0.0)
    h = pd.to_numeric(df.get("hour_of_day", 0), errors="coerce").fillna(0)
    d = pd.to_numeric(df.get("day_of_week", 0), errors="coerce").fillna(0)
    df["hour_sin"] = np.sin(2 * math.pi * h / 24)
    df["hour_cos"] = np.cos(2 * math.pi * h / 24)
    df["dow_sin"] = np.sin(2 * math.pi * d / 7)
    df["dow_cos"] = np.cos(2 * math.pi * d / 7)
    item_v = vocabs["item_vocab"]
    df["recent_product_ids"] = df["session_recent_products"].apply(
        lambda arr: pad_sequence(
            [item_v.get(int(x), 0) for x in arr] if arr is not None and not (isinstance(arr, float) and math.isnan(arr)) else None,
            MAX_RECENT_PRODUCTS,
        )
    )
    df["recent_product_mask"] = df["session_recent_products"].apply(
        lambda arr: make_mask(arr, MAX_RECENT_PRODUCTS)
    )
    brand_v = vocabs["brand_vocab"]
    df["recent_brand_ids"] = df["session_recent_brands"].apply(
        lambda arr: pad_sequence(
            [brand_v.get(x, 0) for x in arr] if arr is not None and not (isinstance(arr, float) and math.isnan(arr)) else None,
            MAX_RECENT_BRANDS,
        )
    )
    df["recent_brand_mask"] = df["session_recent_brands"].apply(
        lambda arr: make_mask(arr, MAX_RECENT_BRANDS)
    )
    return df

def transform_all_features(pdf, vocabs):
    df = transform_item_features(pdf, vocabs)
    return transform_user_features(df, vocabs)

print("Transforms defined (v2)")

In [0]:
# ---- Model-ready column lists ----
ITEM_COLS = [
    "item_idx", "category_idx", "category_l2_idx", "category_l3_idx", "brand_idx",
    "item_log_price", "price_pct_rank", "price_pct_rank_l2",
    "item_popularity", "item_conversion_rate", "item_cart_to_purchase_rate",
    "item_recency_val",
]
USER_NUMERIC_COLS = [
    "session_event_count", "session_avg_price", "session_price_stddev",
    "session_total_value", "session_cart_value",
    "session_view_count", "session_cart_count", "session_purchase_count",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
SEQ_COLS = [
    "recent_product_ids", "recent_product_mask",
    "recent_brand_ids", "recent_brand_mask",
]
MODEL_COLUMNS = ITEM_COLS + USER_NUMERIC_COLS + SEQ_COLS

# ---- Output schema for mapInPandas ----
derived_fields = [
    StructField("item_popularity", DoubleType()),
    StructField("item_conversion_rate", DoubleType()),
    StructField("item_cart_to_purchase_rate", DoubleType()),
    StructField("price_pct_rank", DoubleType()),
    StructField("price_pct_rank_l2", DoubleType()),
    StructField("item_recency_val", DoubleType()),
    StructField("hour_sin", DoubleType()),
    StructField("hour_cos", DoubleType()),
    StructField("dow_sin", DoubleType()),
    StructField("dow_cos", DoubleType()),
    StructField("item_idx", IntegerType()),
    StructField("category_idx", IntegerType()),
    StructField("category_l2_idx", IntegerType()),
    StructField("category_l3_idx", IntegerType()),
    StructField("brand_idx", IntegerType()),
    StructField("recent_product_ids", ArrayType(IntegerType())),
    StructField("recent_product_mask", ArrayType(FloatType())),
    StructField("recent_brand_ids", ArrayType(IntegerType())),
    StructField("recent_brand_mask", ArrayType(FloatType())),
]
output_schema = StructType(list(train_df.schema.fields) + derived_fields)

def make_transform_fn(vocabs):
    def _transform(iterator):
        for pdf in iterator:
            yield transform_all_features(pdf, vocabs)
    return _transform

def _clean_and_write(df, path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    df.to_parquet(path, index=False)

# ---- Transform + shuffle + export ----
for label, sdf, path, shuffle in [
    ("Train", train_df, TRAIN_PARQUET_PATH, True),
    ("Eval",  eval_df,  EVAL_PARQUET_PATH,  False),
    ("Test",  test_df,  TEST_PARQUET_PATH,  False),
]:
    t0 = _time.time()
    transformed = sdf.mapInPandas(make_transform_fn(vocabs), schema=output_schema).select(*MODEL_COLUMNS, "event_time")
    if shuffle:
        transformed = transformed.repartition(NUM_PARQUET_FILES, F.rand(seed=42)).sortWithinPartitions(F.rand(seed=7))
    transformed.write.mode("overwrite").parquet(path)
    n = sum(pq.ParquetFile(f).metadata.num_rows for f in glob.glob(f"{path}/*.parquet"))
    print(f"{label}: {n:,} rows → {path} ({_time.time()-t0:.0f}s)")

# ---- Export lookups ----
_train_items = pd.read_parquet(TRAIN_PARQUET_PATH, columns=ITEM_COLS + ["event_time"])
_eval_idx = pd.read_parquet(EVAL_PARQUET_PATH, columns=["item_idx"])

_item_feat = (_train_items
    .sort_values("event_time", ascending=False)
    .drop_duplicates(subset=["item_idx"], keep="first")
    [ITEM_COLS])
_clean_and_write(_item_feat, f"{VOLUME_BASE}/item_features")
print(f"Item features: {len(_item_feat):,} items")

_tc = _train_items.groupby("item_idx").size().reset_index(name="count")
_clean_and_write(_tc, f"{VOLUME_BASE}/train_item_counts")
print(f"Train counts:  {len(_tc):,} items")

_ei = _eval_idx.drop_duplicates()
_clean_and_write(_ei, f"{VOLUME_BASE}/eval_items")
print(f"Eval items:    {len(_ei):,} unique items")

In [0]:
# ---------------------------------------------------------------------------
# Two-tower model v2: shared embeddings + sequence encoding.
# ---------------------------------------------------------------------------

def masked_mean_pool(embeddings, mask):
    mask_expanded = mask.unsqueeze(-1)
    summed = (embeddings * mask_expanded).sum(dim=1)
    counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
    return summed / counts


class SharedEmbeddings(nn.Module):
    def __init__(self, item_vocab_size, brand_vocab_size, embed_dim):
        super().__init__()
        self.item_emb = nn.Embedding(item_vocab_size, embed_dim, padding_idx=0)
        self.brand_emb = nn.Embedding(brand_vocab_size, embed_dim // 2, padding_idx=0)


class ItemTower(nn.Module):
    def __init__(self, shared_emb, category_vocab_size, category_l2_vocab_size,
                 category_l3_vocab_size, embed_dim=64):
        super().__init__()
        self.shared = shared_emb
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim // 2)
        self.category_l2_emb = nn.Embedding(category_l2_vocab_size, embed_dim // 4)
        self.category_l3_emb = nn.Embedding(category_l3_vocab_size, embed_dim // 4)
        numeric_dim = 7
        mlp_in = embed_dim + (embed_dim // 2) * 2 + (embed_dim // 4) * 2 + numeric_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, embed_dim),
        )

    def forward(self, item_id, category_id, category_l2_id, category_l3_id, brand_id,
                log_price, price_pct_rank, price_pct_rank_l2, item_popularity,
                item_conversion_rate, item_cart_to_purchase_rate, item_recency):
        x = torch.cat([
            self.shared.item_emb(item_id),
            self.category_emb(category_id),
            self.category_l2_emb(category_l2_id),
            self.category_l3_emb(category_l3_id),
            self.shared.brand_emb(brand_id),
            log_price.unsqueeze(-1), price_pct_rank.unsqueeze(-1),
            price_pct_rank_l2.unsqueeze(-1), item_popularity.unsqueeze(-1),
            item_conversion_rate.unsqueeze(-1), item_cart_to_purchase_rate.unsqueeze(-1),
            item_recency.unsqueeze(-1),
        ], dim=-1)
        return TF.normalize(self.mlp(x), p=2, dim=-1)


class UserTower(nn.Module):
    def __init__(self, shared_emb, embed_dim=64):
        super().__init__()
        self.shared = shared_emb
        numeric_dim = 12
        seq_dim = embed_dim + embed_dim // 2
        self.mlp = nn.Sequential(
            nn.Linear(numeric_dim + seq_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, embed_dim),
        )

    def forward(self, numeric, recent_product_ids, recent_product_mask,
                recent_brand_ids, recent_brand_mask):
        prod_emb = self.shared.item_emb(recent_product_ids)
        prod_pooled = masked_mean_pool(prod_emb, recent_product_mask)
        brand_emb = self.shared.brand_emb(recent_brand_ids)
        brand_pooled = masked_mean_pool(brand_emb, recent_brand_mask)
        x = torch.cat([numeric, prod_pooled, brand_pooled], dim=-1)
        return TF.normalize(self.mlp(x), p=2, dim=-1)


class TwoTowerModel(nn.Module):
    def __init__(self, shared_emb, item_tower, user_tower, temperature=0.05):
        super().__init__()
        self.shared_emb = shared_emb
        self.item_tower = item_tower
        self.user_tower = user_tower
        self.temperature = temperature


def create_fresh_model():
    shared = SharedEmbeddings(
        item_vocab_size=len(item_vocab) + 1,
        brand_vocab_size=len(brand_vocab) + 1,
        embed_dim=EMBED_DIM,
    )
    itow = ItemTower(
        shared_emb=shared,
        category_vocab_size=len(category_vocab) + 1,
        category_l2_vocab_size=len(category_l2_vocab) + 1,
        category_l3_vocab_size=len(category_l3_vocab) + 1,
        embed_dim=EMBED_DIM,
    )
    utow = UserTower(shared_emb=shared, embed_dim=EMBED_DIM)
    return TwoTowerModel(shared, itow, utow).to(device)

model = create_fresh_model()
total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Total parameters: {total_params:,}")

In [0]:
# ---------------------------------------------------------------------------
# Training with logQ frequency correction + duplicate masking.
#
# logQ (Yi et al., 2019): subtract log(frequency_j) from each in-batch
# negative's logit to cancel the over-representation of popular items.
# Applied at training time only — inference scores all items uniformly.
# ---------------------------------------------------------------------------
SCALAR_COLS = list({
    "item_idx": "item_id", "category_idx": "category_id",
    "category_l2_idx": "category_l2_id", "category_l3_idx": "category_l3_id",
    "brand_idx": "brand_id", "item_log_price": "log_price",
    "price_pct_rank": "price_pct_rank", "price_pct_rank_l2": "price_pct_rank_l2",
    "item_popularity": "item_popularity", "item_conversion_rate": "item_conversion_rate",
    "item_cart_to_purchase_rate": "item_cart_to_purchase_rate",
    "item_recency_val": "item_recency",
}.keys()) + USER_NUMERIC_COLS

ITEM_COL_MAP = {
    "item_idx": "item_id", "category_idx": "category_id",
    "category_l2_idx": "category_l2_id", "category_l3_idx": "category_l3_id",
    "brand_idx": "brand_id", "item_log_price": "log_price",
    "price_pct_rank": "price_pct_rank", "price_pct_rank_l2": "price_pct_rank_l2",
    "item_popularity": "item_popularity", "item_conversion_rate": "item_conversion_rate",
    "item_cart_to_purchase_rate": "item_cart_to_purchase_rate",
    "item_recency_val": "item_recency",
}


def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)


def train_batches(path, batch_size, seed, epoch):
    files = sorted(glob.glob(f"{path}/*.parquet"))
    random.Random(seed + epoch).shuffle(files)
    all_cols = SCALAR_COLS + SEQ_COLS
    for f in files:
        for rb in pq.ParquetFile(f).iter_batches(batch_size, columns=all_cols):
            batch = {}
            for c in SCALAR_COLS:
                batch[c] = torch.from_numpy(rb.column(c).to_numpy(zero_copy_only=False))
            for c in SEQ_COLS:
                batch[c] = torch.from_numpy(np.stack(rb.column(c).to_pandas().values))
            yield batch


def parquet_batch_to_model_inputs(batch, device):
    item_batch = {}
    for col, kwarg in ITEM_COL_MAP.items():
        dtype = torch.long if col.endswith("idx") else torch.float32
        item_batch[kwarg] = batch[col].to(dtype=dtype, device=device)
    user_numeric = torch.stack(
        [batch[c].float() for c in USER_NUMERIC_COLS], dim=-1
    ).to(device)
    user_batch = {
        "numeric": user_numeric,
        "recent_product_ids": batch["recent_product_ids"].long().to(device),
        "recent_product_mask": batch["recent_product_mask"].float().to(device),
        "recent_brand_ids": batch["recent_brand_ids"].long().to(device),
        "recent_brand_mask": batch["recent_brand_mask"].float().to(device),
    }
    return user_batch, item_batch


# ---- Build log-frequency correction vector ----
_tc = pd.read_parquet(f"{VOLUME_BASE}/train_item_counts")
max_idx = int(_tc['item_idx'].max())
log_freq = torch.zeros(max_idx + 1, device=device)
log_freq[torch.tensor(_tc['item_idx'].values, device=device)] = \
    torch.log(torch.tensor(_tc['count'].values, dtype=torch.float32, device=device))
print(f"logQ vector: {max_idx+1} entries")


def train_logq(model, seed=42, num_epochs=NUM_EPOCHS):
    """Train with logQ correction + duplicate masking."""
    seed_everything(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    history = []
    t0 = _time.time()

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0

        for batch in train_batches(TRAIN_PARQUET_PATH, BATCH_SIZE, seed, epoch):
            if batch[next(iter(batch))].shape[0] < BATCH_SIZE:
                continue
            ub, ib = parquet_batch_to_model_inputs(batch, device)

            u = model.user_tower(**ub)
            i = model.item_tower(**ib)
            logits = u @ i.T / model.temperature

            # logQ correction: subtract log(frequency) for each negative
            ids = ib['item_id']
            logits = logits - log_freq[ids].unsqueeze(0)

            # Mask duplicate items in batch
            dup = (ids.unsqueeze(0) == ids.unsqueeze(1)) & \
                  ~torch.eye(len(ids), dtype=torch.bool, device=device)
            logits = logits.masked_fill(dup, float('-inf'))

            labels = torch.arange(logits.size(0), device=device)
            loss = TF.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

            if n_batches % 50 == 0:
                print(f"  Epoch {epoch}/{num_epochs} — Batch {n_batches}: loss={loss.item():.4f}")

        avg = total_loss / max(n_batches, 1)
        history.append(avg)
        print(f"Epoch {epoch}/{num_epochs}: loss={avg:.4f} ({n_batches} batches, {_time.time()-t0:.0f}s)")

    return history


# ---- Train ----
print(f"Training with logQ correction (seed=42, {NUM_EPOCHS} epochs)...")
train_history = train_logq(model, seed=42)
print(f"\nTraining complete. Final loss: {train_history[-1]:.4f}")

## Evaluation Slices

Every eval/test event is assigned to exactly one slice based on the user's browsing history and whether the target item appears in it:

| Has history? | Target in history? | Slice | What it tests |
| --- | --- | --- | --- |
| Yes | No | **novel\_hist** | Discovery — can the model generalize from past browsing to recommend unseen items? History items are masked out of the candidate set so the model can't cheat. |
| Yes or No | Yes | **repeat** | Re-engagement — can the model recognize a user wants to revisit an item already in their session? High MRR is expected since the shared item embedding provides a strong signal. |
| No | No | **first\_event** | Cold start — with zero user signal, the model relies only on item features and request context (hour, day-of-week). Low MRR reflects the fundamental difficulty of this case. |

**Metrics:** MRR (mean reciprocal rank of the true item), Recall@10, Recall@100 — all computed against the full 154K item catalog.

### Baselines

Three baselines are computed in the same evaluation pass for context:

| Baseline | Method | Applicable slices |
| --- | --- | --- |
| **Global popularity** | Rank every item by training interaction count. The true item's rank = its position in the global popularity list. | All slices |
| **Category popularity** | Same, but scoped to the target item's L1 category. A stronger baseline because it captures within-category demand. | All slices |
| **Recently-viewed** | Rank = position of the true item in `recent_product_ids`. Only meaningful when the item is already in the session history. | repeat only |

The model should substantially beat both popularity baselines on **novel\_hist** (the primary discovery task). On **first\_event** (cold start), the model and category popularity perform similarly — with no user signal, there's little to personalize. On **repeat**, the model slightly outperforms the recently-viewed heuristic because it can distinguish *which* recently-seen item the user is likely to return to.

In [0]:
# ---------------------------------------------------------------------------
# Full-slice evaluation: has-history novel, repeat, first-event.
# Computes model metrics + three baselines in one pass:
#   - Global popularity:   rank by training interaction count
#   - Category popularity: rank within target item's L1 category
#   - Recently-viewed:     position in recent_product_ids (repeat only)
# ---------------------------------------------------------------------------

def build_item_embeddings(model, volume_base):
    """Compute item embeddings for all items in the catalog."""
    item_features = pd.read_parquet(f"{volume_base}/item_features")
    _long = ['item_idx','category_idx','category_l2_idx','category_l3_idx','brand_idx']
    _float = ['item_log_price','price_pct_rank','price_pct_rank_l2',
              'item_popularity','item_conversion_rate','item_cart_to_purchase_rate','item_recency_val']
    model.eval()
    with torch.no_grad():
        args = [torch.tensor(item_features[c].values, dtype=torch.long, device=device) for c in _long]
        args += [torch.tensor(item_features[c].values, dtype=torch.float32, device=device) for c in _float]
        item_embs = TF.normalize(model.item_tower(*args), dim=1)
    ia = item_features['item_idx'].values
    mx = int(ia.max())
    i2p = torch.full((mx+1,), 0, dtype=torch.long, device=device)
    i2p[torch.tensor(ia, device=device)] = torch.arange(len(ia), device=device)
    return item_embs, i2p, ia, mx


def evaluate_all_slices(model, parquet_path, label="eval"):
    """Evaluate model + baselines on has-history novel, repeat, first-event slices.

    Baselines computed in the same pass (no extra data reads):
      - Global popularity:   rank every item by training interaction count
      - Category popularity: rank within the target item's L1 category
      - Recently-viewed:     position in recent_product_ids (repeat slice only)
    """
    item_embs, i2p, ia, mx = build_item_embeddings(model, VOLUME_BASE)
    catalog_size = len(ia)
    ks = [10, 100]
    slices = ['novel_hist', 'repeat', 'first_event']

    # Model accumulators
    mrr = {s: 0.0 for s in slices}
    recall = {s: {k: 0 for k in ks} for s in slices}
    cnt = {s: 0 for s in slices}

    # Baseline accumulators
    pop_mrr = {s: 0.0 for s in slices}
    cat_pop_mrr = {s: 0.0 for s in slices}
    recent_mrr = {s: 0.0 for s in slices}

    # ---- Build popularity lookup tensors ----
    tc = pd.read_parquet(f"{VOLUME_BASE}/train_item_counts")
    tc = tc.sort_values("count", ascending=False).reset_index(drop=True)
    pop_ranked = tc["item_idx"].values
    default_rank = len(pop_ranked) + 1
    pop_rank_t = torch.full((mx + 1,), default_rank, dtype=torch.long, device=device)
    _pi = torch.tensor(pop_ranked, dtype=torch.long, device=device).clamp(0, mx)
    pop_rank_t[_pi] = torch.arange(1, len(pop_ranked) + 1, device=device)

    # ---- Build category-popularity lookup ----
    item_feat = pd.read_parquet(f"{VOLUME_BASE}/item_features")
    cat_map = item_feat.set_index("item_idx")["category_idx"]
    tc_cat = tc.copy()
    tc_cat["cat_idx"] = tc_cat["item_idx"].map(cat_map)
    tc_cat["cat_rank"] = tc_cat.groupby("cat_idx", dropna=False)["count"].rank(
        method="first", ascending=False
    ).astype(int)
    cat_pop_rank_t = torch.full((mx + 1,), default_rank, dtype=torch.long, device=device)
    _ci = torch.tensor(tc_cat["item_idx"].values, dtype=torch.long, device=device).clamp(0, mx)
    _cv = torch.tensor(tc_cat["cat_rank"].values, dtype=torch.long, device=device)
    cat_pop_rank_t[_ci] = _cv

    needed = list(set(SCALAR_COLS + SEQ_COLS))
    for fpath in sorted(glob.glob(f"{parquet_path}/*.parquet")):
        for rb in pq.ParquetFile(fpath).iter_batches(2048, columns=needed):
            iidx = rb.column('item_idx').to_numpy(zero_copy_only=False)
            rp = np.stack(rb.column('recent_product_ids').to_pandas().values)
            rm = np.stack(rb.column('recent_product_mask').to_pandas().values)
            B = len(iidx)
            has_hist = rm.sum(axis=1) > 0
            is_rep = np.zeros(B, dtype=bool)
            for j in range(rp.shape[1]):
                is_rep |= ((rp[:,j]==iidx) & (rm[:,j]>0))
            slice_masks = {
                'novel_hist': (~is_rep) & has_hist,
                'repeat': is_rep,
                'first_event': (~is_rep) & (~has_hist),
            }
            batch = {}
            for c in SCALAR_COLS:
                batch[c] = torch.from_numpy(rb.column(c).to_numpy(zero_copy_only=False))
            for c in SEQ_COLS:
                batch[c] = torch.from_numpy(np.stack(rb.column(c).to_pandas().values))
            ub, _ = parquet_batch_to_model_inputs(batch, device)
            with torch.no_grad():
                ue = model.user_tower(**ub)

            for sl, mask_np in slice_masks.items():
                idx = np.where(mask_np)[0]
                if len(idx) == 0:
                    continue
                _ti = torch.tensor(iidx[idx], dtype=torch.long, device=device)
                _ti_clamped = _ti.clamp(0, mx)
                _ti_pos = i2p[_ti_clamped].clamp(min=0)
                sc = ue[idx] @ item_embs.T
                Bh = len(idx)
                if sl == 'novel_hist':
                    _rp = torch.tensor(rp[idx], dtype=torch.long, device=device)
                    _rm = torch.tensor(rm[idx], dtype=torch.float32, device=device)
                    for j in range(_rp.shape[1]):
                        v = _rm[:,j] > 0; r = torch.where(v)[0]
                        if len(r) > 0:
                            sc[r, i2p[_rp[r,j].clamp(0,mx)].clamp(min=0)] = -float('inf')
                tgt = sc[torch.arange(Bh, device=device), _ti_pos]
                ranks = (sc > tgt.unsqueeze(1)).sum(1) + 1
                mrr[sl] += (1.0 / ranks.float()).sum().item()
                cnt[sl] += Bh
                for k in ks:
                    recall[sl][k] += int((ranks <= k).sum().item())

                # ---- Baselines (computed from lookup tensors, no model) ----
                pop_mrr[sl] += (1.0 / pop_rank_t[_ti_clamped].float()).sum().item()
                cat_pop_mrr[sl] += (1.0 / cat_pop_rank_t[_ti_clamped].float()).sum().item()

                # Recently-viewed: position of true item in recent history
                if sl == 'repeat':
                    _rp_sl = torch.tensor(rp[idx], dtype=torch.long, device=device)
                    _rm_sl = torch.tensor(rm[idx], dtype=torch.float32, device=device)
                    _match = (_rp_sl == _ti.unsqueeze(1)) & (_rm_sl > 0)
                    _first_pos = _match.float().argmax(dim=1)
                    _has_match = _match.any(dim=1)
                    _recent_rank = torch.where(
                        _has_match, _first_pos + 1,
                        torch.tensor(default_rank, dtype=torch.long, device=device),
                    )
                    recent_mrr[sl] += (1.0 / _recent_rank.float()).sum().item()

    # ---- Print results ----
    print(f"\n{'='*90}")
    print(f"  {label.upper()} RESULTS")
    print(f"{'='*90}")
    hdr = (f"  {'Slice':15s} {'Count':>10s}  {'Model':>8s}  {'R@10':>7s}  "
           f"{'R@100':>7s}  {'GlobPop':>8s}  {'CatPop':>8s}  {'Recent':>8s}")
    print(hdr)
    print(f"  {'-'*86}")
    for sl in slices:
        n = cnt[sl]
        if n == 0: continue
        m = mrr[sl] / n
        r10 = recall[sl][10] / n
        r100 = recall[sl][100] / n
        gp = pop_mrr[sl] / n
        cp = cat_pop_mrr[sl] / n
        rc_str = f"{recent_mrr[sl]/n:.4f}" if sl == 'repeat' else "     n/a"
        print(f"  {sl:15s} ({n:>10,})  {m:8.4f}  {r10:7.4f}  {r100:7.4f}  {gp:8.4f}  {cp:8.4f}  {rc_str:>8s}")
    print(f"\n  Random baseline: R@10={10/catalog_size:.6f}, R@100={100/catalog_size:.6f} (catalog={catalog_size:,})")
    print(f"{'='*90}")

    return {s: {'mrr': mrr[s]/max(cnt[s],1), 'r10': recall[s][10]/max(cnt[s],1),
                'r100': recall[s][100]/max(cnt[s],1),
                'pop_mrr': pop_mrr[s]/max(cnt[s],1),
                'cat_pop_mrr': cat_pop_mrr[s]/max(cnt[s],1),
                'recent_mrr': recent_mrr[s]/max(cnt[s],1) if s == 'repeat' else None,
                'count': cnt[s]} for s in slices}


# ---- Run eval ----
model.eval()
eval_results = evaluate_all_slices(model, EVAL_PARQUET_PATH, "Nov 1–3 Eval")

In [0]:
# ---------------------------------------------------------------------------
# Held-out test: single pass on Nov 4–6, no iteration.
# ---------------------------------------------------------------------------
test_results = evaluate_all_slices(model, TEST_PARQUET_PATH, "Nov 4–6 Test")

print(f"\n{'='*70}")
print(f"  SUMMARY: EVAL vs TEST (has-history novel, model only)")
print(f"{'='*70}")
print(f"  {'Metric':15s} {'Eval':>10s}  {'Test':>10s}  {'Δ':>8s}")
for metric in ['mrr', 'r10', 'r100']:
    e = eval_results['novel_hist'][metric]
    t = test_results['novel_hist'][metric]
    print(f"  {metric:15s} {e:10.4f}  {t:10.4f}  {t-e:+8.4f}")
print(f"{'='*70}")

In [0]:
# ---------------------------------------------------------------------------
# Log model + metrics to MLflow.
# Traced user tower and item tower saved as artifacts for serving.
# fe.log_model (with Feature Views) is deferred to a separate deployment step
# that has Feature objects in scope.
# ---------------------------------------------------------------------------
import tempfile, json

with mlflow.start_run(run_name="two-tower-logQ-final") as run:
    # Metrics
    for sl, r in eval_results.items():
        mlflow.log_metric(f"eval_{sl}_mrr", r['mrr'])
        mlflow.log_metric(f"eval_{sl}_r10", r['r10'])
        mlflow.log_metric(f"eval_{sl}_r100", r['r100'])
    for sl, r in test_results.items():
        mlflow.log_metric(f"test_{sl}_mrr", r['mrr'])
        mlflow.log_metric(f"test_{sl}_r10", r['r10'])
        mlflow.log_metric(f"test_{sl}_r100", r['r100'])
    # Optional blend metrics (only if RRF notebook was run)
    try:
        mlflow.log_metric("eval_blend_mrr", eval_blend['blend_mrr'])
        mlflow.log_metric("test_blend_mrr", test_blend['blend_mrr'])
    except NameError:
        pass
    for i, loss in enumerate(train_history):
        mlflow.log_metric(f"train_loss_epoch{i+1}", loss)

    # Params
    mlflow.log_param("embed_dim", EMBED_DIM)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("num_epochs", NUM_EPOCHS)
    mlflow.log_param("temperature", 0.05)
    mlflow.log_param("logq_correction", True)
    mlflow.log_param("dup_masking", True)
    mlflow.log_param("seed", 42)
    mlflow.log_param("total_params", total_params)

    # Traced model artifacts
    model.eval()
    _artifact_dir = tempfile.mkdtemp()
    _dummy_numeric = torch.randn(1, 12).to(device)
    _dummy_prod_ids = torch.zeros(1, MAX_RECENT_PRODUCTS, dtype=torch.long).to(device)
    _dummy_prod_mask = torch.ones(1, MAX_RECENT_PRODUCTS).to(device)
    _dummy_brand_ids = torch.zeros(1, MAX_RECENT_BRANDS, dtype=torch.long).to(device)
    _dummy_brand_mask = torch.ones(1, MAX_RECENT_BRANDS).to(device)
    traced_user = torch.jit.trace(
        model.user_tower,
        (_dummy_numeric, _dummy_prod_ids, _dummy_prod_mask, _dummy_brand_ids, _dummy_brand_mask),
    )
    traced_user.save(f"{_artifact_dir}/user_tower_traced.pt")
    traced_item = torch.jit.trace(model.item_tower, (
        torch.tensor([0], dtype=torch.long).to(device),
        torch.tensor([0], dtype=torch.long).to(device),
        torch.tensor([0], dtype=torch.long).to(device),
        torch.tensor([0], dtype=torch.long).to(device),
        torch.tensor([0], dtype=torch.long).to(device),
        torch.tensor([0.0]).to(device), torch.tensor([0.5]).to(device),
        torch.tensor([0.5]).to(device), torch.tensor([0.0]).to(device),
        torch.tensor([0.1]).to(device), torch.tensor([0.1]).to(device),
        torch.tensor([0.0]).to(device),
    ))
    traced_item.save(f"{_artifact_dir}/item_tower_traced.pt")
    mlflow.log_artifacts(_artifact_dir, artifact_path="traced_towers")

    # Vocabs
    with open(f"{_artifact_dir}/vocabs.json", "w") as f:
        json.dump({k: {str(kk): vv for kk, vv in v.items()} for k, v in vocabs.items()}, f)
    mlflow.log_artifact(f"{_artifact_dir}/vocabs.json")

    print(f"MLflow run: {run.info.run_id}")
    print(f"  Eval novel-hist MRR: {eval_results['novel_hist']['mrr']:.4f}")
    print(f"  Test novel-hist MRR: {test_results['novel_hist']['mrr']:.4f}")
    print(f"  Artifacts: traced user tower, item tower, vocabs.json")